# Phase 5 — LSI: decoder 126M AdaLoRA


In [1]:
%pip install -q huggingface_hub==2.3.0 tokenizers scikit-learn tqdm peft==0.21.2 transformers==5.20.2


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.3.0 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 2.3.0 which is incompatible.
gradio-client 2.6.1 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 2.3.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import copy
import os, json, math, random, gc, tempfile, zipfile, uuid, re, hashlib
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download, HfFileSystem
from sklearn.metrics import precision_recall_fscore_support, f1_score, accuracy_score
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
DATA_REPO = 'Exploration-Lab/CS-781-Capstone'
BUCKET_ROOT = 'hf://buckets/A-240031/legal-lm-phase1-bucket'
PHASE1_TAG = 'decoder_126M'
MODEL_PATH = 'checkpoints/decoder_126M/model.pt'
TOKENIZER_PATH = 'tokenizer_126M_decoder/tokenizer.json'
MODEL_CONFIG = {'d_model': 1024, 'n_heads': 8, 'n_kv_heads': 4, 'n_layers': 8, 'd_ff': 2816, 'max_seq_len': 2048}
MAX_SEQ_LEN = 2048
CHUNK_OVERLAP = 64  # number of shared tokens, NOT the stride
CHUNK_ENCODE_BATCH_SIZE = 4  # chunks per backbone call
USE_GRADIENT_CHECKPOINTING = False  # recompute each chunk microbatch during backward
UNFREEZE_LAST_N_BLOCKS = 0  # frozen backbone; increase only for partial full fine-tuning
TRAIN_MAX_CHUNKS = 2
EVAL_MAX_CHUNKS = 4
USE_AMP = DEVICE == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16

def autocast_context():
    return torch.autocast(device_type=DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP)

DATA_PREFIX = 'Phase_5_LSI'
LABEL_KEY = 'labels'
assert 0 < MAX_SEQ_LEN <= MODEL_CONFIG.get('max_seq_len', MODEL_CONFIG.get('max_encoder_len'))
assert 0 <= CHUNK_OVERLAP < MAX_SEQ_LEN
assert CHUNK_ENCODE_BATCH_SIZE > 0
assert UNFREEZE_LAST_N_BLOCKS == 0
BATCH_SIZE = 2  # complete cases per DataLoader batch
GRAD_ACCUM_STEPS = 4
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
NUM_EPOCHS = 5
PATIENCE = 10
MIN_DELTA = 0.001
ADAPTER_CONFIG = dict(init_r=16,target_r=8,alpha=32,dropout=.05,orth_reg_weight=.1)
HEAD_DROPOUT = 0.1
UPLOAD_CHECKPOINTS = True
TECHNIQUE = 'adalora'
TRAIN_FRACTION = 0.4  # use all LSI training records
POS_WEIGHT_MAX = 10.0
PER_LABEL_MIN_SUPPORT = 5
THRESHOLD_GRID = [i / 100 for i in range(5, 81, 5)]

# All loading/saving locations are controlled here.
# Paths below are relative to BUCKET_ROOT; an absolute hf:// path also works.
SAVE_TAG = f'lsi_{PHASE1_TAG}_{TECHNIQUE}_partial_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}'  # None generates a descriptive tag; or enter your own run tag.
SUBMISSION_ZIP_NAME = None  # None uses SAVE_TAG + '.zip'; or enter a filename.
CHECKPOINT_PREFIX = 'checkpoints'
RESUME = True  # True resumes SAVE_TAG; set SAVE_TAG to the existing latest tag.
WORK_ROOT = Path('/kaggle/working/lsi_runs') if Path('/kaggle/working').exists() else Path.cwd() / 'lsi_runs'

if RESUME and SAVE_TAG is None:
    raise ValueError('To resume, set SAVE_TAG to the existing run tag.')
if SAVE_TAG is None:
    SAVE_TAG = f'lsi_{PHASE1_TAG}_{TECHNIQUE}_partial_ov{CHUNK_OVERLAP}_sample{TRAIN_MAX_CHUNKS}_eval{EVAL_MAX_CHUNKS}'
if not isinstance(SAVE_TAG, str) or not re.fullmatch(r'[A-Za-z0-9_.-]+', SAVE_TAG) or SAVE_TAG in {'.', '..'}:
    raise ValueError('SAVE_TAG must be a single name containing letters, numbers, _, - or .')
BEST_TAG = SAVE_TAG + '_best'
if SUBMISSION_ZIP_NAME is None:
    SUBMISSION_ZIP_NAME = SAVE_TAG + '.zip'
if Path(SUBMISSION_ZIP_NAME).name != SUBMISSION_ZIP_NAME or not SUBMISSION_ZIP_NAME.endswith('.zip'):
    raise ValueError('SUBMISSION_ZIP_NAME must be a filename ending in .zip.')
WORK_DIR = WORK_ROOT / SAVE_TAG
WORK_DIR.mkdir(parents=True, exist_ok=True)

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF')
    except Exception:
        pass
fs = HfFileSystem(token=HF_TOKEN)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
seed_everything(SEED)
print('Device:', DEVICE)
def bucket_path(path):
    if path.startswith('hf://'):
        return path
    return BUCKET_ROOT.rstrip('/') + '/' + path.lstrip('/')

def checkpoint_remote_path(tag):
    return bucket_path(f"{CHECKPOINT_PREFIX.strip('/')}/{tag}/checkpoint.pt")

def download_bucket_file(remote_path):
    # Identity-specific cache directory; refresh the bytes on each call.
    remote = bucket_path(remote_path)
    cache_key = hashlib.sha256(remote.encode()).hexdigest()[:20]
    local = WORK_ROOT / '_inputs' / cache_key / remote.rsplit('/', 1)[-1]
    local.parent.mkdir(parents=True, exist_ok=True)
    temporary = local.with_name(local.name + '.download')
    fs.get_file(remote, str(temporary))
    os.replace(temporary, local)
    return local

def assert_run_destination():
    if RESUME:
        return
    for tag in [SAVE_TAG, BEST_TAG]:
        local = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
        if local.exists() or (UPLOAD_CHECKPOINTS and fs.exists(checkpoint_remote_path(tag))):
            raise FileExistsError(
                f'Checkpoint already exists for {tag}. Choose a new SAVE_TAG, '
                'or set RESUME=True with the original latest SAVE_TAG.'
            )
    manifest = dict(save_tag=SAVE_TAG, best_tag=BEST_TAG, technique=TECHNIQUE,
                    phase1_tag=PHASE1_TAG, model_path=MODEL_PATH,
                    tokenizer_path=TOKENIZER_PATH, submission_zip=SUBMISSION_ZIP_NAME,
                    train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                    use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                    max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP, input_format='overlapping_chunks_v1',
                    pooling='token_mean_then_chunk_mean',
                    unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS)
    with open(WORK_DIR / 'run_config.json', 'w') as f:
        json.dump(manifest, f, indent=2)

print('Phase-1 model:', bucket_path(MODEL_PATH))
print('Tokenizer:', bucket_path(TOKENIZER_PATH))
print('Latest checkpoint:', checkpoint_remote_path(SAVE_TAG))
print('Best checkpoint:', checkpoint_remote_path(BEST_TAG))
print('Submission ZIP:', WORK_DIR / SUBMISSION_ZIP_NAME)

print('Chunk length/overlap:', MAX_SEQ_LEN, CHUNK_OVERLAP)


Device: cuda
Phase-1 model: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/decoder_126M/model.pt
Tokenizer: hf://buckets/A-240031/legal-lm-phase1-bucket/tokenizer_126M_decoder/tokenizer.json
Latest checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4/checkpoint.pt
Best checkpoint: hf://buckets/A-240031/legal-lm-phase1-bucket/checkpoints/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4_best/checkpoint.pt
Submission ZIP: /kaggle/working/lsi_runs/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4.zip
Chunk length/overlap: 2048 64


In [3]:
local_tokenizer = download_bucket_file(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(str(local_tokenizer))
tokenizer.no_padding()
tokenizer.no_truncation()
VOCAB_SIZE = tokenizer.get_vocab_size()
EMPTY_ID = tokenizer.token_to_id('<|eos|>')
PAD_ID = tokenizer.token_to_id('<|pad|>')
if EMPTY_ID is None:
    raise ValueError('Expected the Phase-1 <|eos|> token.')
if PAD_ID is None: PAD_ID = EMPTY_ID


def download_json(filename):
    path = hf_hub_download(repo_id=DATA_REPO, filename=filename, repo_type='dataset', token=HF_TOKEN)
    with open(path, encoding='utf-8') as f: return json.load(f)

def normalize_documents(raw, labelled):
    if isinstance(raw, list): docs = raw
    elif isinstance(raw, dict) and 'id' in raw and 'text' in raw: docs = [raw]
    elif isinstance(raw, dict):
        docs = []
        for doc_id, value in raw.items():
            if not isinstance(value, dict): raise ValueError('Expected ID -> case object mapping')
            doc = dict(value)
            doc.setdefault('id', doc_id)
            if doc['id'] != doc_id: raise ValueError('Document ID mismatch')
            docs.append(doc)
    else: raise ValueError('Expected a list of cases or ID -> case mapping')
    result = []
    for row_index, doc in enumerate(docs):
        case_id, text = doc['id'], doc['text']
        if not isinstance(text, str): raise ValueError(f'{case_id}: text must be a string')
        row = dict(id=case_id, text=text, row_index=row_index)
        if labelled:
            if LABEL_KEY not in doc:
                raise ValueError(f'Missing label field {LABEL_KEY!r}; available fields: {list(doc)}. Set LABEL_KEY.')
            label = doc[LABEL_KEY]
            if not isinstance(label, list) or not label or any(type(x) is not int for x in label):
                raise ValueError(f'{case_id}: expected nonempty list of integer statute IDs, got {label!r}')
            row['labels'] = sorted(set(label))
        result.append(row)
    if not result: raise ValueError('Empty split')
    return result

# Keep each input record separate, even when IDs repeat.
# Test is loaded later, only for prediction.
data = {}
for split in ['train', 'dev']:
    data[split] = normalize_documents(
        download_json(f'{DATA_PREFIX}/Train/{split}.json'), labelled=True)
    counts = Counter(d['id'] for d in data[split])
    print(split, '| records:', len(data[split]), '| unique IDs:', len(counts),
          '| repeated IDs:', sum(n > 1 for n in counts.values()))

# IDs are opaque integers: never assume contiguous IDs or an index offset.
LABELS = sorted({label for split in ('train', 'dev') for d in data[split] for label in d['labels']})
LABEL_TO_INDEX = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)
if not NUM_LABELS: raise ValueError('No statute labels')
if set(d['id'] for d in data['train']) & set(d['id'] for d in data['dev']):
    raise ValueError('Train/dev case-ID overlap; resolve before training')
for split in data:
    if len({d['id'] for d in data[split]}) != len(data[split]):
        raise ValueError(f'{split}: duplicate case IDs; resolve explicitly before training')
print('Statutes:', NUM_LABELS, '| IDs:', LABELS)


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B / 2.47MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

train | records: 42750 | unique IDs: 42750 | repeated IDs: 0


Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

dev | records: 10181 | unique IDs: 10181 | repeated IDs: 0
Statutes: 100 | IDs: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99]


In [4]:
# Sample by case; preserve at least one training example for each observed statute.
assert 0 < TRAIN_FRACTION <= 1
full_train = data['train']
rng = random.Random(SEED)
order = list(range(len(full_train))); rng.shuffle(order)
selected = set(order[:max(1, round(len(order) * TRAIN_FRACTION))])
covered = {label for i in selected for label in full_train[i]['labels']}
for i in order:
    if any(label not in covered for label in full_train[i]['labels']):
        selected.add(i); covered.update(full_train[i]['labels'])
data['train'] = [d for i, d in enumerate(full_train) if i in selected]
TRAIN_LABEL_COUNTS = torch.tensor([sum(label in d['labels'] for d in data['train']) for label in LABELS], dtype=torch.float)
POS_WEIGHT = ((len(data['train']) - TRAIN_LABEL_COUNTS) / TRAIN_LABEL_COUNTS.clamp_min(1)).sqrt().clamp(1, POS_WEIGHT_MAX)
POS_WEIGHT[TRAIN_LABEL_COUNTS == 0] = 1
print('Training records:', len(full_train), '->', len(data['train']))
print('Train support per statute:', dict(zip(LABELS, TRAIN_LABEL_COUNTS.int().tolist())))
print('Dev-only statutes (no supervised train positives):', [l for l,c in zip(LABELS,TRAIN_LABEL_COUNTS) if c == 0])
del full_train


Training records: 42750 -> 17100
Train support per statute: {0: 152, 1: 420, 2: 142, 3: 182, 4: 135, 5: 4702, 6: 285, 7: 428, 8: 137, 9: 312, 10: 1620, 11: 250, 12: 1754, 13: 1677, 14: 1524, 15: 107, 16: 257, 17: 322, 18: 154, 19: 193, 20: 197, 21: 283, 22: 193, 23: 127, 24: 192, 25: 128, 26: 768, 27: 125, 28: 250, 29: 201, 30: 1295, 31: 456, 32: 149, 33: 599, 34: 4164, 35: 1240, 36: 229, 37: 481, 38: 665, 39: 2851, 40: 208, 41: 326, 42: 230, 43: 3104, 44: 1471, 45: 1016, 46: 967, 47: 360, 48: 226, 49: 160, 50: 145, 51: 1248, 52: 419, 53: 517, 54: 880, 55: 729, 56: 234, 57: 184, 58: 565, 59: 122, 60: 110, 61: 1458, 62: 169, 63: 656, 64: 265, 65: 131, 66: 128, 67: 445, 68: 239, 69: 265, 70: 523, 71: 920, 72: 366, 73: 221, 74: 125, 75: 203, 76: 198, 77: 1606, 78: 669, 79: 160, 80: 233, 81: 511, 82: 240, 83: 408, 84: 110, 85: 666, 86: 144, 87: 234, 88: 625, 89: 798, 90: 798, 91: 540, 92: 134, 93: 231, 94: 1521, 95: 161, 96: 773, 97: 3119, 98: 191, 99: 254}
Dev-only statutes (no supervised

In [5]:
def chunk_token_ids(tokens, max_len=MAX_SEQ_LEN, overlap=CHUNK_OVERLAP):
    if not 0 <= overlap < max_len: raise ValueError('Invalid chunk overlap')
    if not tokens: return [[EMPTY_ID]]
    chunks = []
    for start in range(0, len(tokens), max_len - overlap):
        chunks.append(tokens[start:start + max_len])
        if start + max_len >= len(tokens): break
    return chunks

def select_chunk_indices(count, limit, training, seed):
    if limit is None or count <= limit:
        return list(range(count))
    if limit < 1: raise ValueError('Chunk limit must be positive or None')
    if training:
        rng = random.Random(seed)
        # One random chunk from each equal-sized region; two regions = document halves.
        return [rng.randrange(i * count // limit, (i + 1) * count // limit)
                for i in range(limit)]
    if limit == 1: return [count // 2]
    return [i * (count - 1) // (limit - 1) for i in range(limit)]

class LSICaseDataset(Dataset):
    def __init__(self, documents, split):
        self.split, self.epoch = split, 0
        self.rows = []
        token_lengths, chunk_counts = [], []
        for doc in tqdm(documents, desc='Tokenizing cases'):
            tokens = tokenizer.encode(doc['text'], add_special_tokens=False).ids
            chunks = chunk_token_ids(tokens)
            self.rows.append(dict(doc_id=doc['id'], chunks=chunks, labels=doc.get('labels', [])))
            token_lengths.append(len(tokens)); chunk_counts.append(len(chunks))
        lengths = torch.tensor(token_lengths, dtype=torch.float)
        print('Cases/chunks:', len(self.rows), sum(chunk_counts), 'max chunks/case:', max(chunk_counts))
        print('Token lengths p50/p90/p99/max:', torch.quantile(lengths, torch.tensor([.5,.9,.99,1.])).tolist())
    def __len__(self): return len(self.rows)
    def set_epoch(self, epoch): self.epoch = epoch
    def __getitem__(self, index):
        row = self.rows[index]
        training = self.split == 'train'
        limit = TRAIN_MAX_CHUNKS if training else EVAL_MAX_CHUNKS
        selected = select_chunk_indices(len(row['chunks']), limit, training,
                                        SEED + self.epoch * len(self.rows) + index)
        return dict(row, chunks=[row['chunks'][i] for i in selected])

def collate_lsi(rows):
    chunks, owners = [], []
    for case_index, row in enumerate(rows):
        chunks.extend(row['chunks'])
        owners.extend([case_index] * len(row['chunks']))
    lengths = torch.tensor([len(c) for c in chunks], dtype=torch.long)
    ids = torch.full((len(chunks), int(lengths.max())), PAD_ID, dtype=torch.long)
    for i, chunk in enumerate(chunks): ids[i, :len(chunk)] = torch.tensor(chunk)
    return dict(input_ids=ids, lengths=lengths, chunk_to_case=torch.tensor(owners, dtype=torch.long),
                labels=torch.tensor([[float(label in r['labels']) for label in LABELS] for r in rows], dtype=torch.float),
                doc_ids=[r['doc_id'] for r in rows])

datasets = {split: LSICaseDataset(docs, split) for split, docs in data.items()}
def make_loader(split):
    return DataLoader(datasets[split], batch_size=BATCH_SIZE, shuffle=(split == 'train'),
                      collate_fn=collate_lsi, num_workers=0, pin_memory=(DEVICE == 'cuda'))


Tokenizing cases:   0%|          | 0/17100 [00:00<?, ?it/s]

Cases/chunks: 17100 25560 max chunks/case: 62
Token lengths p50/p90/p99/max: [628.0, 4114.0, 11999.2734375, 121735.0]


Tokenizing cases:   0%|          | 0/10181 [00:00<?, ?it/s]

Cases/chunks: 10181 16190 max chunks/case: 59
Token lengths p50/p90/p99/max: [797.0, 4487.0, 13458.435546875, 115496.0]


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# RoPE
# ============================================================

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_seq_len=2048, base=10000):
        super().__init__()

        inv_freq = 1.0 / (
            base ** (torch.arange(0, head_dim, 2).float() / head_dim)
        )

        positions = torch.arange(max_seq_len).float()
        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos(), persistent=False)
        self.register_buffer("sin", freqs.sin(), persistent=False)

    def forward(self, q, k):
        # q, k: [B, H, T, D]
        T = q.size(2)

        cos = self.cos[:T].unsqueeze(0).unsqueeze(0)
        sin = self.sin[:T].unsqueeze(0).unsqueeze(0)

        q1, q2 = q[..., ::2], q[..., 1::2]
        k1, k2 = k[..., ::2], k[..., 1::2]

        q_rot = torch.stack(
            [q1 * cos - q2 * sin,
             q1 * sin + q2 * cos],
            dim=-1,
        ).flatten(-2)

        k_rot = torch.stack(
            [k1 * cos - k2 * sin,
             k1 * sin + k2 * cos],
            dim=-1,
        ).flatten(-2)

        return q_rot, k_rot


# ============================================================
# SwiGLU
# ============================================================

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()

        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)

    def forward(self, x):
        return self.down_proj(
            F.silu(self.gate_proj(x)) * self.up_proj(x)
        )


# ============================================================
# GQA + SDPA
# ============================================================

class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len=2048):
        super().__init__()

        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0

        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = d_model // n_heads

        self.q_proj = nn.Linear(
            d_model,
            n_heads * self.head_dim,
            bias=False,
        )

        self.k_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.v_proj = nn.Linear(
            d_model,
            n_kv_heads * self.head_dim,
            bias=False,
        )

        self.out_proj = nn.Linear(
            d_model,
            d_model,
            bias=False,
        )

        self.rope = RotaryEmbedding(
            self.head_dim,
            max_seq_len=max_seq_len,
        )

    def forward(self, x):

        B, T, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(
            B, T,
            self.n_heads,
            self.head_dim
        ).transpose(1, 2)

        k = k.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        v = v.view(
            B, T,
            self.n_kv_heads,
            self.head_dim
        ).transpose(1, 2)

        q, k = self.rope(q, k)

        # Repeat KV heads for GQA
        repeat_factor = self.n_heads // self.n_kv_heads

        k = k.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        v = v.repeat_interleave(
            repeat_factor,
            dim=1,
        )

        # PyTorch SDPA can dispatch to Flash Attention
        # / memory-efficient kernels automatically.
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True,
        )

        attn_out = (
            attn_out
            .transpose(1, 2)
            .contiguous()
            .view(B, T, -1)
        )

        return self.out_proj(attn_out)


# ============================================================
# Decoder Block
# ============================================================

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model,
        n_heads,
        n_kv_heads,
        d_ff,
        max_seq_len=2048,
    ):
        super().__init__()

        self.norm1 = nn.RMSNorm(d_model)
        self.attention = GQAAttention(
            d_model,
            n_heads,
            n_kv_heads,
            max_seq_len=max_seq_len,
        )

        self.norm2 = nn.RMSNorm(d_model)
        self.ffn = SwiGLU(
            d_model,
            d_ff,
        )

    def forward(self, x):

        x = x + self.attention(
            self.norm1(x)
        )

        x = x + self.ffn(
            self.norm2(x)
        )

        return x


# ============================================================
# Decoder-only Language Model
# ============================================================

class EfficientDecoderLM(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_kv_heads=4,
        n_layers=12,
        d_ff=3072,
        max_seq_len=2048,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        self.token_embedding = nn.Embedding(                             
            vocab_size,
            d_model,
        )

        self.blocks = nn.ModuleList([
            DecoderBlock(
                d_model=d_model,
                n_heads=n_heads,
                n_kv_heads=n_kv_heads,
                d_ff=d_ff,
                max_seq_len=max_seq_len,
            )
            for _ in range(n_layers)
        ])

        self.final_norm = nn.RMSNorm(d_model)

        # Tied LM head
        self.lm_head = nn.Linear(
            d_model,
            vocab_size,
            bias=False,
        )

        self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

    def _init_weights(self, module):

        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.Embedding):
            nn.init.xavier_uniform_(module.weight)

        elif isinstance(module, nn.RMSNorm):
            nn.init.ones_(module.weight)

    def forward(self, input_ids):

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)


In [7]:
from torch.utils.checkpoint import checkpoint as activation_checkpoint

class LSIModel(EfficientDecoderLM):
    def __init__(self, vocab_size, head_dropout=0.1, **kwargs):
        super().__init__(vocab_size=vocab_size, **kwargs)
        del self.lm_head
        self.lsi_dropout = nn.Dropout(head_dropout)
        self.lsi_head = nn.Linear(self.d_model, NUM_LABELS)

    def encode_chunks(self, input_ids, lengths):
        x = self.token_embedding(input_ids)
        for block in self.blocks: x = block(x)
        x = self.final_norm(x)
        # Right padding cannot affect preceding real tokens under causal attention.
        mask = torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]
        return (x.float() * mask.unsqueeze(-1)).sum(1) / lengths[:, None].float()

    def forward(self, input_ids, lengths, chunk_to_case, num_cases):
        device = self.lsi_head.weight.device
        pooled_chunks = []
        for start in range(0, input_ids.size(0), CHUNK_ENCODE_BATCH_SIZE):
            end = min(start + CHUNK_ENCODE_BATCH_SIZE, input_ids.size(0))
            lens = lengths[start:end].to(device, non_blocking=True)
            # Trim this microbatch to its own longest chunk.
            width = int(lengths[start:end].max())
            ids = input_ids[start:end, :width].to(device, non_blocking=True)
            if self.training and USE_GRADIENT_CHECKPOINTING:
                h = activation_checkpoint(self.encode_chunks, ids, lens, use_reentrant=False,
                                          preserve_rng_state=True)
            else: h = self.encode_chunks(ids, lens)
            pooled_chunks.append(h)
        h = torch.cat(pooled_chunks, dim=0)
        owners = chunk_to_case.to(device, non_blocking=True)
        sums = h.new_zeros((num_cases, self.d_model)).index_add(0, owners, h)
        counts = torch.bincount(owners, minlength=num_cases).to(h.dtype).unsqueeze(1)
        assert bool((counts > 0).all()), 'Every case must have at least one chunk'
        document_vectors = sums / counts
        return self.lsi_head(self.lsi_dropout(document_vectors))

def forward_batch(model, batch):
    with autocast_context():
        return model(batch['input_ids'], batch['lengths'], batch['chunk_to_case'], len(batch['doc_ids']))

phase1_path = download_bucket_file(MODEL_PATH)

def phase1_state():
    checkpoint = torch.load(phase1_path, map_location='cpu', weights_only=False)
    if isinstance(checkpoint, dict):
        for key in ('model_state_dict', 'model', 'state_dict'):
            if key in checkpoint: checkpoint = checkpoint[key]; break
    if not isinstance(checkpoint, dict) or not all(isinstance(v, torch.Tensor) for v in checkpoint.values()):
        raise ValueError('Cannot identify Phase-1 state dict')
    state = dict(checkpoint)
    state.pop('lm_head.weight', None)
    return state

def load_phase1(model):
    missing, unexpected = model.load_state_dict(phase1_state(), strict=False)
    assert set(missing) == {'lsi_head.weight','lsi_head.bias'} and not unexpected, (missing,unexpected)
    return model

def parameter_report(model):
    print('Total/trainable parameters:',sum(p.numel() for p in model.parameters()),
          sum(p.numel() for p in model.parameters() if p.requires_grad))
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2,6):
    raise RuntimeError('AdaLoRA dependencies require PyTorch >= 2.6; use a current Kaggle GPU runtime.')
from peft import AdaLoraConfig, AdaLoraModel
import peft

TOTAL_OPTIMIZER_STEPS = NUM_EPOCHS * math.ceil(len(make_loader('train')) / GRAD_ACCUM_STEPS)

def build_lsi_model():
    model = load_phase1(LSIModel(VOCAB_SIZE,head_dropout=HEAD_DROPOUT,**MODEL_CONFIG))
    targets = [f'blocks.{i}.attention.{p}' for i in range(MODEL_CONFIG['n_layers']-2,MODEL_CONFIG['n_layers'])
               for p in ('q_proj','k_proj','v_proj','out_proj')]
    config = AdaLoraConfig(init_r=ADAPTER_CONFIG['init_r'],target_r=ADAPTER_CONFIG['target_r'],
        lora_alpha=ADAPTER_CONFIG['alpha'],lora_dropout=ADAPTER_CONFIG['dropout'],
        target_modules=targets,total_step=TOTAL_OPTIMIZER_STEPS,
        tinit=int(.10*TOTAL_OPTIMIZER_STEPS),tfinal=int(.20*TOTAL_OPTIMIZER_STEPS),
        deltaT=10,beta1=.85,beta2=.85,orth_reg_weight=ADAPTER_CONFIG['orth_reg_weight'],bias='none')
    # Direct tuner wrapper preserves the custom forward signature and allocation API.
    model = AdaLoraModel(model,config,adapter_name='default').to(DEVICE)
    for p in model.model.lsi_head.parameters(): p.requires_grad=True
    parameter_report(model)
    print('AdaLoRA steps/init/final:',TOTAL_OPTIMIZER_STEPS,config.tinit,config.tfinal)
    return model

def training_loss(model,batch,logits,y,teacher=None):
    supervised = F.binary_cross_entropy_with_logits(logits.float(),y,pos_weight=POS_WEIGHT.to(DEVICE),reduction='none').mean(-1).sum()
    # Custom forward returns logits, so apply AdaLoRA's orthogonality loss explicitly.
    regularizers=[]
    for name,p in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            p32=p.float()
            gram=p32@p32.T if 'lora_A' in name else p32.T@p32
            regularizers.append(torch.linalg.matrix_norm(gram-torch.eye(gram.size(0),device=gram.device)))
    if regularizers:
        supervised = supervised + len(y)*ADAPTER_CONFIG['orth_reg_weight']*torch.stack(regularizers).mean()
    return supervised

def after_optimizer_step(model,global_step):
    # Called after a successful optimizer step while the gradients still exist.
    model.update_and_allocate(global_step)

def adaptation_state(model):
    allocator=model.rankallocator
    return dict(rank_pattern=copy.deepcopy(model.peft_config['default'].rank_pattern),
        allocator={key:{name:t.detach().cpu() for name,t in getattr(allocator,key).items()}
                   for key in ('ipt','exp_avg_ipt','exp_avg_unc')})

def restore_adaptation_state(model,state):
    if not state:return
    model.peft_config['default'].rank_pattern=state['rank_pattern']
    for key,values in state['allocator'].items():
        setattr(model.rankallocator,key,{name:t.to(DEVICE) for name,t in values.items()})


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B /  515MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

In [8]:
ADAPTATION_DETAILS = dict(peft='0.21.2',total_optimizer_steps=TOTAL_OPTIMIZER_STEPS,target_blocks='last2',tinit_fraction=.1,tfinal_fraction=.2,deltaT=10)

def checkpoint_model_state(model):
    if TECHNIQUE=='distillation': return {k:v.detach().cpu() for k,v in model.state_dict().items()}
    names={name for name,p in model.named_parameters() if p.requires_grad}
    return {k:v.detach().cpu() for k,v in model.state_dict().items()
            if k in names or (TECHNIQUE=='adalora' and ('lora_' in k or 'ranknum' in k))}

def restore_model_state(model,payload):
    if payload.get('model_format')=='full':
        model.load_state_dict(payload['model'],strict=True);return
    names={name for name,p in model.named_parameters() if p.requires_grad}
    assert names.issubset(payload['model']), 'Missing trainable weights'
    _,unexpected=model.load_state_dict(payload['model'],strict=False)
    assert not unexpected, unexpected
def experiment_metadata(domain='LSI'):
    return dict(domain=domain, phase1_tag=PHASE1_TAG, model_path=MODEL_PATH, tokenizer_path=TOKENIZER_PATH,
                labels=LABELS, model_config=MODEL_CONFIG, adapter_config=ADAPTER_CONFIG,
                adaptation_details=ADAPTATION_DETAILS,
                max_seq_len=MAX_SEQ_LEN, chunk_overlap=CHUNK_OVERLAP,
                chunk_encode_batch_size=CHUNK_ENCODE_BATCH_SIZE,
                gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
                train_max_chunks=TRAIN_MAX_CHUNKS, eval_max_chunks=EVAL_MAX_CHUNKS,
                sampling='stratified_epoch_seed_v1', use_amp=USE_AMP, amp_dtype=str(AMP_DTYPE),
                unfreeze_last_n_blocks=UNFREEZE_LAST_N_BLOCKS,
                pooling='token_mean_then_chunk_mean', input_format='overlapping_chunks_v1',
                data_repo=DATA_REPO, data_prefix=DATA_PREFIX, label_key=LABEL_KEY,
                technique=TECHNIQUE, head_dropout=HEAD_DROPOUT,
                train_fraction=TRAIN_FRACTION, pos_weight=POS_WEIGHT.tolist(), threshold_grid=THRESHOLD_GRID,
                per_label_min_support=PER_LABEL_MIN_SUPPORT, seed=SEED, batch_size=BATCH_SIZE, grad_accum_steps=GRAD_ACCUM_STEPS,
                lr=LR, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                patience=PATIENCE, min_delta=MIN_DELTA)

def save_lsi_checkpoint(tag, model, optimizer, epoch, extra, domain):
    folder = WORK_DIR / 'checkpoints' / tag
    folder.mkdir(parents=True, exist_ok=True)
    rng = {'python': random.getstate(), 'torch': torch.get_rng_state()}
    if torch.cuda.is_available(): rng['cuda'] = torch.cuda.get_rng_state_all()
    payload = dict(model=checkpoint_model_state(model), model_format=('full' if TECHNIQUE=='distillation' else 'trainable_only'),
                   adaptation_state=adaptation_state(model),optimizer=optimizer.state_dict(), rng=rng,
                   epoch=epoch, extra=extra, metadata=experiment_metadata(domain))
    tmp = folder / 'checkpoint.tmp'
    torch.save(payload, tmp)
    os.replace(tmp, folder / 'checkpoint.pt')
    if UPLOAD_CHECKPOINTS:
        fs.put_file(str(folder / 'checkpoint.pt'), checkpoint_remote_path(tag))

def load_lsi_checkpoint(tag, model, domain, optimizer=None, restore_rng=False):
    path = WORK_DIR / 'checkpoints' / tag / 'checkpoint.pt'
    if not path.exists() or restore_rng:
        if UPLOAD_CHECKPOINTS:
            path = download_bucket_file(checkpoint_remote_path(tag))
        elif not path.exists():
            raise FileNotFoundError(path)
    payload = torch.load(path, map_location='cpu', weights_only=False)  # own trusted checkpoints only
    assert payload['metadata'] == experiment_metadata(domain), 'Checkpoint configuration mismatch'
    restore_model_state(model,payload)
    restore_adaptation_state(model,payload.get('adaptation_state',{}))
    if optimizer is not None: optimizer.load_state_dict(payload['optimizer'])
    if restore_rng:
        random.setstate(payload['rng']['python'])
        torch.set_rng_state(payload['rng']['torch'])
        if torch.cuda.is_available() and 'cuda' in payload['rng']:
            torch.cuda.set_rng_state_all(payload['rng']['cuda'])
    return payload

@torch.no_grad()
def evaluate_lsi(model, loader, thresholds=None, tune=True):
    model.eval()
    targets, scores = [], []
    total_loss = total_n = 0
    for batch in loader:
        y = batch['labels'].to(DEVICE)
        logits = forward_batch(model, batch).float()
        total_loss += F.binary_cross_entropy_with_logits(logits, y, pos_weight=POS_WEIGHT.to(DEVICE), reduction='none').mean(-1).sum().item()
        total_n += len(y)
        targets.append(y.cpu()); scores.append(logits.sigmoid().cpu())
    y = torch.cat(targets).numpy().astype('int64')
    p = torch.cat(scores).numpy()
    # Select on dev only, with the same >= comparison used during test prediction.
    if tune:
        global_t = max(THRESHOLD_GRID, key=lambda t: (f1_score(y, p >= t, average='macro', zero_division=0), -abs(t-.5)))
        thresholds = [global_t] * NUM_LABELS
        for j in range(NUM_LABELS):
            support = int(y[:, j].sum())
            if PER_LABEL_MIN_SUPPORT <= support < len(y):
                candidates = sorted(set(THRESHOLD_GRID + [global_t]))
                thresholds[j] = max(candidates, key=lambda t: (f1_score(y[:,j], p[:,j]>=t, zero_division=0), -abs(t-global_t)))
    if thresholds is None: thresholds = [.5] * NUM_LABELS
    pred = p >= thresholds
    precision, recall, f1, support = precision_recall_fscore_support(y, pred, average=None, zero_division=0)
    return dict(loss=total_loss/total_n, macro_f1=float(f1.mean()),
                micro_f1=float(f1_score(y,pred,average='micro',zero_division=0)),
                subset_accuracy=float(accuracy_score(y,pred)),
                macro_precision=float(precision.mean()), macro_recall=float(recall.mean()),
                thresholds=[float(t) for t in thresholds],
                per_label={int(label):dict(precision=float(precision[j]),recall=float(recall[j]),f1=float(f1[j]),support=int(support[j])) for j,label in enumerate(LABELS)})
def cleanup_latest_checkpoint():
    # Exact file deletion only. Never delete a parent directory or a Phase-1 file.
    best_local = WORK_DIR / 'checkpoints' / BEST_TAG / 'checkpoint.pt'
    if not best_local.is_file(): raise RuntimeError('Best checkpoint is missing; keeping latest')
    if UPLOAD_CHECKPOINTS:
        best_remote = checkpoint_remote_path(BEST_TAG)
        if not fs.exists(best_remote) or fs.info(best_remote)['size'] != best_local.stat().st_size:
            raise RuntimeError('Best upload not verified; keeping latest')
        latest_remote = checkpoint_remote_path(SAVE_TAG)
        assert latest_remote != best_remote and latest_remote != bucket_path(MODEL_PATH)
        if fs.exists(latest_remote): fs.rm(latest_remote)
        if fs.exists(latest_remote): raise RuntimeError('Latest bucket checkpoint deletion failed')
    latest_local = WORK_DIR / 'checkpoints' / SAVE_TAG / 'checkpoint.pt'
    if latest_local.exists(): latest_local.unlink()
    print('Training completed: retained best checkpoint only:', BEST_TAG)


In [9]:
def train_lsi(domain='LSI'):
    assert domain == 'LSI'
    assert_run_destination()
    seed_everything(SEED)
    model = build_lsi_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    tag = SAVE_TAG
    best_tag = BEST_TAG
    start = 0
    global_step = 0
    extra = dict(best_f1=-1.0, patience_reference=-1.0, stale=0, history=[], best_tag=best_tag)
    if RESUME:
        checkpoint = load_lsi_checkpoint(SAVE_TAG, model, domain, optimizer, restore_rng=True)
        start, extra = checkpoint['epoch'], checkpoint['extra']
        global_step = extra.get('global_step',0)
        best_tag = extra['best_tag']
        if 'scaler' in extra: scaler.load_state_dict(extra['scaler'])
        del checkpoint
    train_loader, dev_loader = make_loader('train'), make_loader('dev')
    assert len(train_loader) > 0
    teacher = build_teacher() if TECHNIQUE=='distillation' else None
    for epoch in range(start, NUM_EPOCHS):
        if extra['stale'] >= PATIENCE: break
        datasets['train'].set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        train_sum = train_n = group_n = 0
        for step, batch in enumerate(tqdm(train_loader, desc=f'{domain} epoch {epoch+1}')):
            y = batch['labels'].to(DEVICE)
            logits = forward_batch(model, batch)
            loss_sum = training_loss(model,batch,logits,y,teacher)
            scaler.scale(loss_sum).backward()
            group_n += len(y)
            train_n += len(y)
            train_sum += loss_sum.item()
            if (step + 1) % GRAD_ACCUM_STEPS == 0 or step + 1 == len(train_loader):
                scaler.unscale_(optimizer)
                for p in params:
                    if p.grad is not None: p.grad.div_(group_n)
                torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                old_scale = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() >= old_scale:
                    global_step += 1
                    after_optimizer_step(model,global_step)
                optimizer.zero_grad(set_to_none=True)
                group_n = 0
        metrics = evaluate_lsi(model, dev_loader)
        f1 = metrics['macro_f1']
        improved = f1 > extra['best_f1']
        if improved:
            extra['best_f1'] = f1
            extra['thresholds'] = metrics['thresholds']
        if f1 > extra['patience_reference'] + MIN_DELTA:
            extra['patience_reference'], extra['stale'] = f1, 0
        else: extra['stale'] += 1
        extra['history'].append(dict(epoch=epoch+1, train_loss=train_sum/train_n, **metrics))
        print(f"LSI train loss={train_sum/train_n:.4f}, dev macro-F1={f1:.4f}, precision={metrics['macro_precision']:.4f}, recall={metrics['macro_recall']:.4f}, patience={extra['stale']}/{PATIENCE}")
        extra['scaler'] = scaler.state_dict()
        extra['global_step'] = global_step
        if improved: save_lsi_checkpoint(best_tag, model, optimizer, epoch+1, extra, domain)
        save_lsi_checkpoint(tag, model, optimizer, epoch+1, extra, domain)
    best_payload = load_lsi_checkpoint(best_tag, model, domain)
    model.thresholds = best_payload['extra']['thresholds']
    del best_payload
    metrics = evaluate_lsi(model, dev_loader, thresholds=model.thresholds, tune=False)
    with open(WORK_DIR / f'{domain}_dev_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
    with open(WORK_DIR / f'{domain}_history.json', 'w') as f: json.dump(extra['history'], f, indent=2)
    print('Best development metrics:', metrics)
    with open(WORK_DIR / 'label_thresholds.json', 'w') as f:
        json.dump(dict(labels=LABELS, thresholds=model.thresholds), f, indent=2)
    cleanup_latest_checkpoint()
    del optimizer, teacher
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return model, extra['history']


In [10]:
def validate_predictions(predictions, documents):
    assert len(predictions) == len(documents)
    assert [p['id'] for p in predictions] == [d['id'] for d in documents]
    assert len({p['id'] for p in predictions}) == len(predictions)
    assert all(set(p) == {'id','pred'} and isinstance(p['pred'],list) and len(p['pred']) == len(set(p['pred'])) and all(type(x) is int and x in LABEL_TO_INDEX for x in p['pred']) for p in predictions)

@torch.no_grad()
def predict_lsi(model):
    model.eval()
    predictions = []
    for batch in tqdm(make_loader('test'), desc='LSI test predictions'):
        scores = forward_batch(model, batch).float().sigmoid().cpu()
        selected = scores >= torch.tensor(model.thresholds)[None, :]
        predictions.extend(dict(id=doc_id, pred=[int(LABELS[j]) for j in row.nonzero().flatten().tolist()]) for doc_id,row in zip(batch['doc_ids'],selected))
    validate_predictions(predictions, data['test'])
    return predictions

lsi_model, lsi_history = train_lsi()
# Load test only after training. Its IDs are unique in the supplied data.
data['test'] = normalize_documents(
    download_json(f'{DATA_PREFIX}/Test/test.json'), labelled=False)
datasets['test'] = LSICaseDataset(data['test'], 'test')
predictions = predict_lsi(lsi_model)
json_path = WORK_DIR / 'prediction.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)
del lsi_model
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


Total/trainable parameters: 128991468 332004
AdaLoRA steps/init/final: 10690 1069 2138


hf://buckets/A-240031/legal-lm-phase1-bu(…): reconstructing file:   0%|          |  0.00B / 4.06MB            

hf://buckets/A-240031/legal-lm-phase1-bu(…): downloading bytes:           |  0.00B            

LSI epoch 5:   0%|          | 0/8550 [00:00<?, ?it/s]

LSI train loss=0.3051, dev macro-F1=0.3017, precision=0.2732, recall=0.3765, patience=0/10


Processing Files (0 / 0): |          |  0.00B /  0.00B            

... (7 outputs left out)

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            

Tokenizing cases:   0%|          | 0/13019 [00:00<?, ?it/s]

Cases/chunks: 13019 20354 max chunks/case: 112
Token lengths p50/p90/p99/max: [786.0, 4267.0, 13114.5, 220937.0]


LSI test predictions:   0%|          | 0/6510 [00:00<?, ?it/s]

In [11]:
# Packaging can be rerun without retraining.
json_path = WORK_DIR / 'prediction.json'
with open(json_path, encoding='utf-8') as f: predictions = json.load(f)
validate_predictions(predictions, data['test'])
zip_path = WORK_DIR / SUBMISSION_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(json_path, arcname='prediction.json')
print('JSON:', json_path)
print('ZIP:', zip_path)
print('Cases:', len(predictions))


JSON: /kaggle/working/lsi_runs/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4/prediction.json
ZIP: /kaggle/working/lsi_runs/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4/lsi_decoder_126M_adalora_partial_ov64_sample2_eval4.zip
Cases: 13019
